In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/DeepLearningProject/data/train/task2train540p /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision import datasets
from torch.utils.data import DataLoader, random_split
import os

torch.backends.cudnn.benchmark = True

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
data_path = "/content/task2train540p"

full_dataset = datasets.ImageFolder(data_path)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_indices, val_indices = random_split(
    range(len(full_dataset)),
    [train_size, val_size]
)

train_dataset = datasets.ImageFolder(
    data_path,
    transform=train_transforms
)

val_dataset = datasets.ImageFolder(
    data_path,
    transform=val_transforms
)

train_dataset = torch.utils.data.Subset(train_dataset, train_indices.indices)
val_dataset = torch.utils.data.Subset(val_dataset, val_indices.indices)

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_features = resnet.fc.in_features
num_classes = len(full_dataset.classes)

resnet.fc = nn.Linear(num_features, num_classes)

In [ ]:
for param in resnet.parameters():
    param.requires_grad = False

for param in resnet.layer3.parameters():
    param.requires_grad = True

for param in resnet.layer4.parameters():
    param.requires_grad = True

for param in resnet.fc.parameters():
    param.requires_grad = True

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = resnet.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet.parameters()),
    lr=1e-4
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)


In [ ]:
num_epochs = 20
best_val_accuracy = 0
patience = 3
counter = 0

for epoch in range(num_epochs):

    # ---- TRAIN ----
    resnet.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = resnet(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)

    # ---- VALIDATION ----
    resnet.eval()
    val_correct = 0
    val_total = 0
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_accuracy = 100 * val_correct / val_total
    val_loss = val_loss / len(val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.2f}% "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.2f}%")

    # ---- Save Best Model ----
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(resnet.state_dict(), "best_resnet_model.pt")
        print("🔥 Best model saved!")
        counter = 0
    else:
        counter += 1

    # ---- Early Stopping ----
    if counter >= patience:
        print("⛔ Early stopping triggered")
        break

    scheduler.step()

print("Training complete!")
print(f"Best Validation Accuracy: {best_val_accuracy:.2f}%")

Epoch [1/20] Train Loss: 1.1566 | Train Acc: 46.62% Val Loss: 0.9197 | Val Acc: 63.00%
🔥 Best model saved!
Epoch [2/20] Train Loss: 0.7048 | Train Acc: 73.75% Val Loss: 0.6489 | Val Acc: 72.00%
🔥 Best model saved!
Epoch [3/20] Train Loss: 0.4200 | Train Acc: 87.75% Val Loss: 0.5454 | Val Acc: 78.00%
🔥 Best model saved!
Epoch [4/20] Train Loss: 0.3004 | Train Acc: 92.38% Val Loss: 0.5203 | Val Acc: 79.50%
🔥 Best model saved!
Epoch [5/20] Train Loss: 0.2622 | Train Acc: 93.75% Val Loss: 0.5119 | Val Acc: 80.50%
🔥 Best model saved!
Epoch [6/20] Train Loss: 0.2813 | Train Acc: 93.25% Val Loss: 0.4918 | Val Acc: 83.00%
🔥 Best model saved!
Epoch [7/20] Train Loss: 0.2446 | Train Acc: 94.75% Val Loss: 0.4830 | Val Acc: 83.00%
Epoch [8/20] Train Loss: 0.2326 | Train Acc: 95.62% Val Loss: 0.4975 | Val Acc: 83.00%
Epoch [9/20] Train Loss: 0.2248 | Train Acc: 95.12% Val Loss: 0.4829 | Val Acc: 83.50%
🔥 Best model saved!
Epoch [10/20] Train Loss: 0.2122 | Train Acc: 95.38% Val Loss: 0.4854 | Val A